In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.ensemble import GradientBoostingClassifier 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
import warnings
warnings.filterwarnings("ignore")

In [18]:
import torch.cuda as cuda
device = torch.device('cuda' if cuda.is_available() else 'cpu')

In [19]:
df = pd.read_csv('f1_results_2025.csv')

In [20]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished


In [21]:
df['final_position'] = pd.to_numeric(df['final_position'], errors='coerce')

In [22]:
df

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,2025,17,Azerbaijan Grand Prix,2025-09-21,Baku City Circuit,Nico Hülkenberg,German,Sauber,17,16,0.0,51,+1:20.237,1:44.370,Finished
335,2025,17,Azerbaijan Grand Prix,2025-09-21,Baku City Circuit,Lance Stroll,Canadian,Aston Martin,14,17,0.0,51,+1:36.392,1:45.083,Finished
336,2025,17,Azerbaijan Grand Prix,2025-09-21,Baku City Circuit,Pierre Gasly,French,Alpine F1 Team,18,18,0.0,50,+3.865,1:45.492,Lapped
337,2025,17,Azerbaijan Grand Prix,2025-09-21,Baku City Circuit,Franco Colapinto,Argentine,Alpine F1 Team,16,19,0.0,50,+6.325,1:46.055,Lapped


In [23]:
preferred = [
    'fastest_lap'
]
auto = [c for c in df.columns if 'lap' in c.lower() and ('fast' in c.lower() or 'fastest' in c.lower())]
for p in preferred:
    if p in df.columns and p not in auto:
        auto.append(p)
col = auto[0] if auto else None


s = (df[col]
        .astype('string')
        .fillna('')
        .str.strip()
        .str.lower()
        .str.replace(' ', '', regex=False))

s = s.str.replace(r'(?<=\d)m(?=\d)', ':', regex=True)
s = s.str.replace(r's$', '', regex=True)

colon_count = s.str.count(':')
with_colon = s.where(colon_count.gt(0))
base = with_colon.fillna('')

# Convert timedeltas to seconds without using .dt (avoid TimedeltaIndex .dt error)
td = pd.to_timedelta(
    np.where(colon_count.eq(1), '00:' + base, base),
    errors='coerce'
)
secs_colon = (td / pd.Timedelta(seconds=1)).astype(float)

secs_no_colon = pd.to_numeric(s.where(colon_count.eq(0)), errors='coerce')

mask = colon_count.gt(0).to_numpy()
a = np.asarray(secs_colon, dtype=float)
b = np.asarray(secs_no_colon, dtype=float)

df['fastest_lap_seconds'] = np.where(mask, a, b).astype('float64')
parsed = int(np.isfinite(df['fastest_lap_seconds']).sum())
total = len(df)
print(f"Created df['fastest_lap_seconds'] from column '{col}' ({parsed}/{total} parsed).")

Created df['fastest_lap_seconds'] from column 'fastest_lap' (322/339 parsed).


In [24]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished,84.597


In [25]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished,84.597


In [26]:
df.tail(60)

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
279,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Oscar Piastri,Australian,McLaren,1,1,25.0,72,1:38:29.849,1:12.271,Finished,72.271
280,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Max Verstappen,Dutch,Red Bull,3,2,18.0,72,+1.271,1:12.921,Finished,72.921
281,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Isack Hadjar,French,RB F1 Team,4,3,15.0,72,+3.233,1:13.327,Finished,73.327
282,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,George Russell,British,Mercedes,5,4,12.0,72,+5.654,1:13.728,Finished,73.728
283,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Alexander Albon,Thai,Williams,15,5,10.0,72,+6.327,1:13.687,Finished,73.687
284,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Oliver Bearman,British,Haas F1 Team,20,6,8.0,72,+9.044,1:13.950,Finished,73.950
285,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Lance Stroll,Canadian,Aston Martin,19,7,6.0,72,+9.497,1:13.822,Finished,73.822
286,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Fernando Alonso,Spanish,Aston Martin,10,8,4.0,72,+11.709,1:13.719,Finished,73.719
287,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Yuki Tsunoda,Japanese,Red Bull,12,9,2.0,72,+13.597,1:14.354,Finished,74.354
288,2025,15,Dutch Grand Prix,2025-08-31,Circuit Park Zandvoort,Esteban Ocon,French,Haas F1 Team,18,10,1.0,72,+14.063,1:13.986,Finished,73.986


In [27]:

cand = lambda *xs: next((c for c in xs if c in df.columns), None)
col_driver = cand('driver_name', 'driver', 'Driver', 'driverId', 'driver_id')
col_final  = cand('final_position', 'position', 'Result')
col_points = cand('points', 'Points')
col_date   = cand('race_date', 'date', 'Date')
col_season = cand('season', 'Year')
col_round  = cand('round', 'Round')
col_race   = cand('race_name', 'grand_prix', 'Grand Prix')

if col_driver is None or col_final is None:
    raise KeyError('Required columns not found: driver and final_position')

work = df.copy()
work[col_final] = pd.to_numeric(work[col_final], errors='coerce')
if col_points:
    work[col_points] = pd.to_numeric(work[col_points], errors='coerce')

# Ensure season is available (derive from date if needed)
if col_season is None and col_date:
    work[col_date] = pd.to_datetime(work[col_date], errors='coerce', dayfirst=True)
    work['__season_tmp__'] = work[col_date].dt.year
    col_season = '__season_tmp__'

# Identify 2025 grid drivers
if col_season is None:
    raise KeyError('No season/year information available to identify 2025 grid drivers.')

grid_mask = work[col_season] == 2025
grid_drivers = (
    work.loc[grid_mask & work[col_driver].notna(), col_driver]
        .dropna()
        .unique()
        .tolist()
)

# Nothing to compute if none found
if not grid_drivers:
    raise ValueError('No drivers found for season 2025 in the dataset.')

# Sort chronologically
if col_date:
    # If date col exists but not yet parsed
    if not np.issubdtype(work[col_date].dtype, np.datetime64):
        work[col_date] = pd.to_datetime(work[col_date], errors='coerce', dayfirst=True)
    sort_cols = [col_date]
    if col_race:
        sort_cols.append(col_race)
elif col_season and col_round:
    sort_cols = [col_season, col_round]
else:
    sort_cols = [work.index]

work = work.sort_values(sort_cols).reset_index(drop=True)

# Keep only 2025 grid drivers
work = work[work[col_driver].isin(grid_drivers)].copy()

# Take last 3 races per driver
by = work[col_driver]
last3 = work.groupby(by, group_keys=False).tail(5)

# Aggregate form metrics
form_2025 = last3.groupby(col_driver).agg(
    races=(col_final, 'count'),
    avg_finish=(col_final, 'mean'),
    median_finish=(col_final, 'median'),
    best_finish=(col_final, 'min'),
    top10_rate=(col_final, lambda s: np.mean((s <= 10).astype(float))),
    podium_rate=(col_final, lambda s: np.mean((s <= 3).astype(float))),
    win_rate=(col_final, lambda s: np.mean((s == 1).astype(float))),
)
if col_points:
    pts = last3.groupby(col_driver)[col_points].agg(avg_points='mean', sum_points='sum')
    form_2025 = form_2025.join(pts)

# Order: best average finish, then best finish
form_2025 = form_2025.sort_values(['avg_finish', 'avg_points'])

print(f"Current form for 2025 grid (last 3 GPs): Drivers = {len(form_2025)}")
form_2025

Current form for 2025 grid (last 3 GPs): Drivers = 21


,races,avg_finish,median_finish,best_finish,top10_rate,podium_rate,win_rate,avg_points,sum_points
driver_name,,,,,,,,,
Max Verstappen,5,3.4,2.0,1,1.0,0.6,0.4,16.4,82.0
George Russell,5,3.8,4.0,2,1.0,0.4,0.0,13.0,65.0
Oscar Piastri,5,5.4,2.0,1,0.8,0.8,0.4,16.6,83.0
Lando Norris,5,6.0,2.0,1,0.8,0.6,0.2,13.4,67.0
Charles Leclerc,5,7.8,4.0,3,0.8,0.2,0.0,8.2,41.0
Alexander Albon,5,9.2,7.0,5,0.6,0.0,0.0,4.8,24.0
Liam Lawson,5,9.4,8.0,5,0.6,0.0,0.0,3.6,18.0
Gabriel Bortoleto,5,9.8,9.0,6,0.6,0.0,0.0,2.8,14.0
Lewis Hamilton,5,10.6,8.0,6,0.6,0.0,0.0,3.6,18.0


In [28]:

features = df.copy()

driver_form = form_2025.reset_index()


In [29]:
# Split data for training/testing
from sklearn.model_selection import train_test_split

# Define X (features) and y (target - race winner or position)
X = features.drop(['final_position', 'driver_name', 'race_name', 'race_date'], axis=1)
y = features[['final_position','grid_position']]  # or binary target (1 for winner, 0 for non-winner)

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

In [30]:
X.head()


,season_year,race_round_number,circuit_name,driver_nationality,constructor_name,grid_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Albert Park Grand Prix Circuit,British,McLaren,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Albert Park Grand Prix Circuit,Dutch,Red Bull,3,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Albert Park Grand Prix Circuit,British,Mercedes,4,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Albert Park Grand Prix Circuit,Italian,Mercedes,16,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Albert Park Grand Prix Circuit,Thai,Williams,6,10.0,57,+12.773,1:24.597,Finished,84.597


In [33]:


# =========================
# STEP 2: Load Data
# =========================
# reload to ensure fresh state (safe in notebook)
df = pd.read_csv("f1_results_2025.csv")

# Ensure race_date is datetime
df["race_date"] = pd.to_datetime(df.get("race_date", pd.Series()), errors="coerce")

# =========================
# STEP 3: Feature Engineering
# =========================
def build_features(df):
    features = []
    # require columns
    req = ["driver_name", "race_date", "final_position", "status", "grid_position", "race_name"]
    for c in req:
        if c not in df.columns:
            raise KeyError(f"Required column missing: {c}")

    for driver in df["driver_name"].dropna().unique():
        driver_df = df[df["driver_name"] == driver].sort_values("race_date")
        # Need at least 5 prior races; first index with 5-history available is i=5 (0-based)
        for i in range(5, len(driver_df)):
            past_races = driver_df.iloc[i-5:i]
            if len(past_races) < 5:
                continue
            race = driver_df.iloc[i]

            driver_form = past_races["final_position"].astype(float).mean()
            driver_dnf_rate = (past_races["status"].str.contains("DNF", case=False, na=False)).mean()

            features.append({
                "driver_name": race["driver_name"],
                "race_name": race["race_name"],
                "race_date": race["race_date"],
                "grid_position": pd.to_numeric(race.get("grid_position", np.nan), errors="coerce"),
                "driver_form": driver_form,
                "driver_dnf_rate": float(driver_dnf_rate),
                "target_win": 1 if int(race.get("final_position", 999)) == 1 else 0,
                "target_podium": 1 if int(race.get("final_position", 999)) <= 3 else 0
            })
    return pd.DataFrame(features)

feat_df = build_features(df)

if feat_df.empty:
    print("No feature rows created (not enough history). Check races per driver.")
else:
    print(f"Built {len(feat_df)} training rows.")

# =========================
# STEP 4: Train Models
# =========================
def train_models(feat_df):
    X = feat_df[["grid_position", "driver_form", "driver_dnf_rate"]].copy()
    y_win = feat_df["target_win"].copy()
    y_pod = feat_df["target_podium"].copy()

    # Use index-based split so both targets/models share the exact same train/test rows
    idx_train, idx_test = train_test_split(feat_df.index, test_size=0.2, random_state=42, stratify=y_win if y_win.nunique()>1 else None)

    X_train = X.loc[idx_train]
    X_test = X.loc[idx_test]

    y_win_train = y_win.loc[idx_train]
    y_win_test = y_win.loc[idx_test]

    y_pod_train = y_pod.loc[idx_train]
    y_pod_test = y_pod.loc[idx_test]

    # Fill or impute simple NaNs (median for grid, mean for continuous)
    grid_median = X_train['grid_position'].median()
    X_train['grid_position'] = X_train['grid_position'].fillna(grid_median)
    X_test['grid_position'] = X_test['grid_position'].fillna(grid_median)

    form_mean = X_train['driver_form'].mean()
    X_train['driver_form'] = X_train['driver_form'].fillna(form_mean)
    X_test['driver_form'] = X_test['driver_form'].fillna(form_mean)

    X_train['driver_dnf_rate'] = X_train['driver_dnf_rate'].fillna(0.0)
    X_test['driver_dnf_rate'] = X_test['driver_dnf_rate'].fillna(0.0)

    model_win = XGBClassifier(

                                use_label_encoder=False,
    eval_metric='logloss',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    scale_pos_weight=1.0,   # compute as (n_negative / n_positive) for imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=1)
    model_win.fit(X_train, y_win_train)

    model_pod = XGBClassifier(
                                use_label_encoder=False,
    eval_metric='logloss',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    scale_pos_weight=1.0,   # compute as (n_negative / n_positive) for imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=1)
    model_pod.fit(X_train, y_pod_train)

    print("Winner model accuracy:", accuracy_score(y_win_test, model_win.predict(X_test)))
    print("Podium model accuracy:", accuracy_score(y_pod_test, model_pod.predict(X_test)))

    return model_win, model_pod

model_win, model_pod = (None, None)
if not feat_df.empty:
    model_win, model_pod = train_models(feat_df)

# =========================
# STEP 5: Predict Future Race
# =========================
def predict_future_race(df, race_name, race_date, model_win, model_pod, grid_dict=None, grid_weight: float = 0.3):
    """
    Predict per-driver win & podium probabilities for a future race.
    grid_weight controls how strongly qualifying position shifts win odds (pole highest).
    """
    race_date = pd.to_datetime(race_date)
    past_df = df[df["race_date"] < race_date]

    drivers = past_df["driver_name"].dropna().unique()
    features = []

    for driver in drivers:
        driver_history = past_df[past_df["driver_name"] == driver].sort_values("race_date").tail(5)
        if driver_history.empty:
            continue

        driver_form = driver_history["final_position"].astype(float).mean()
        driver_dnf_rate = (driver_history["status"].str.contains("DNF", case=False, na=False)).mean()

        if grid_dict and driver in grid_dict:
            grid_position = grid_dict[driver]
        else:
            grid_position = driver_history["grid_position"].dropna().mean()

        features.append({
            "driver_name": driver,
            "driver_form": driver_form,
            "driver_dnf_rate": float(driver_dnf_rate),
            "grid_position": float(grid_position) if not pd.isna(grid_position) else np.nan,
        })

    feat_df = pd.DataFrame(features)
    if feat_df.empty:
        print("No prediction rows - check past data and race_date")
        return feat_df

    # simple imputation
    feat_df['grid_position'] = feat_df['grid_position'].fillna(feat_df['grid_position'].median())
    feat_df['driver_form'] = feat_df['driver_form'].fillna(feat_df['driver_form'].median())
    feat_df['driver_dnf_rate'] = feat_df['driver_dnf_rate'].fillna(0.0)

    X = feat_df[["grid_position", "driver_form", "driver_dnf_rate"]]

    if model_win is not None:
        feat_df["win_prob"] = model_win.predict_proba(X)[:, 1]
    else:
        feat_df["win_prob"] = np.nan

    if model_pod is not None:
        feat_df["podium_prob"] = model_pod.predict_proba(X)[:, 1]
    else:
        feat_df["podium_prob"] = np.nan

    # Normalize and apply grid-based prior to win probabilities
    try:
        feat_df['win_prob_raw'] = feat_df['win_prob']
        eps = 1e-6
        p = feat_df['win_prob_raw'].clip(eps, 1 - eps)
        logits = np.log(p / (1 - p))
        # Grid bonus: pole (1) gets +grid_weight, P2 gets +(grid_weight * (1 - 1)), etc.
        # Using linear decay: bonus = -grid_weight * (grid_position - 1)
        grid_bonus = -grid_weight * (feat_df['grid_position'] - 1.0)
        adj_logits = logits + grid_bonus.values
        exps = np.exp(adj_logits - adj_logits.max())
        feat_df['win_prob'] = exps / exps.sum()
        feat_df['win_prob_norm'] = feat_df['win_prob']
        feat_df.drop('win_prob_raw', axis=1, inplace=True)
        print(f"Sum of win_prob after grid-adjusted normalization: {feat_df['win_prob'].sum():.6f}")
    except Exception as e:
        print("Warning: failed to normalize with grid prior:", e)

    return feat_df.sort_values("win_prob", ascending=False)

# =========================
# STEP 6: Example - Predict Hungarian GP 2025 (adjust as needed)
# =========================
if model_win is not None:
    preds = predict_future_race(df, "Italian GP", "2025-10-06", model_win, model_pod, grid_dict=None, grid_weight=0.5)
    print("\nTop Predictions for Italian GP 2025 (grid-weighted):")
    print(preds.head())
else:
    print("Models not trained; skipping example prediction.")


Built 234 training rows.
Winner model accuracy: 0.9574468085106383
Podium model accuracy: 1.0
Sum of win_prob after grid-adjusted normalization: 1.000000

Top Predictions for Italian GP 2025 (grid-weighted):
       driver_name  driver_form  driver_dnf_rate  grid_position  win_prob  \
1   Max Verstappen          3.4              0.0            3.4  0.785857   
0     Lando Norris          6.0              0.0            3.0  0.114805   
8    Oscar Piastri          5.4              0.0            3.4  0.076183   
2   George Russell          3.8              0.0            5.0  0.014905   
7  Charles Leclerc          7.8              0.0            4.8  0.007268   

   podium_prob  win_prob_norm  
1     0.274886       0.785857  
0     0.260160       0.114805  
8     0.839705       0.076183  
2     0.401177       0.014905  
7     0.435031       0.007268  
Winner model accuracy: 0.9574468085106383
Podium model accuracy: 1.0
Sum of win_prob after grid-adjusted normalization: 1.000000

Top Pre

In [35]:
# =========================
# STEP 7: PyTorch Model (5 Hidden Layers, Adam) for Win & Podium
# =========================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Tuple

# Reuse feat_df built earlier; if empty, skip
if 'feat_df' not in globals() or feat_df.empty:
    print("feat_df missing or empty; build features first before training PyTorch model.")
else:
    # Select feature columns and targets
    torch_features = feat_df[["grid_position", "driver_form", "driver_dnf_rate"]].copy()
    torch_targets_win = feat_df["target_win"].astype(float).values
    torch_targets_pod = feat_df["target_podium"].astype(float).values

    # Basic numeric scaling (standardization)
    feat_mean = torch_features.mean()
    feat_std = torch_features.std().replace(0, 1.0)
    torch_features_std = (torch_features - feat_mean) / feat_std

    class RaceDataset(Dataset):
        def __init__(self, X: np.ndarray, y_win: np.ndarray, y_pod: np.ndarray):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y_win = torch.tensor(y_win, dtype=torch.float32).unsqueeze(1)
            self.y_pod = torch.tensor(y_pod, dtype=torch.float32).unsqueeze(1)
        def __len__(self):
            return self.X.shape[0]
        def __getitem__(self, idx):
            return self.X[idx], self.y_win[idx], self.y_pod[idx]

    # Train/validation split (80/20)
    idx = np.arange(len(torch_features_std))
    rng = np.random.default_rng(42)
    rng.shuffle(idx)
    split = int(0.8 * len(idx))
    idx_train, idx_val = idx[:split], idx[split:]

    X_train_np = torch_features_std.iloc[idx_train].values
    X_val_np = torch_features_std.iloc[idx_val].values
    y_train_win = torch_targets_win[idx_train]
    y_val_win = torch_targets_win[idx_val]
    y_train_pod = torch_targets_pod[idx_train]
    y_val_pod = torch_targets_pod[idx_val]

    train_ds = RaceDataset(X_train_np, y_train_win, y_train_pod)
    val_ds = RaceDataset(X_val_np, y_val_win, y_val_pod)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class RaceNet(nn.Module):
        def __init__(self, in_dim: int):
            super().__init__()
            # 5 hidden layers (add activations if needed)
            self.net = nn.Sequential(
                nn.Linear(in_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 128),
                nn.ReLU(),
                nn.Linear(128, 256),
                nn.ReLU(),
                nn.Linear(256, 512),
                nn.ReLU(),
                nn.Linear(512, 16),
                nn.ReLU(),
            )
            # Dual heads
            self.head_win = nn.Linear(16, 1)
            self.head_pod = nn.Linear(16, 1)
        def forward(self, x):
            h = self.net(x)
            return self.head_win(h), self.head_pod(h)

    model_torch = RaceNet(in_dim=torch_features_std.shape[1]).to(device)

    # Loss & optimizer
    bce = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model_torch.parameters(), lr=1e-3, weight_decay=1e-4)

    def evaluate(model: nn.Module, loader: DataLoader) -> Tuple[float, float, float, float]:
        model.eval()
        total = 0
        win_correct = 0
        pod_correct = 0
        win_loss_cum = 0.0
        pod_loss_cum = 0.0
        with torch.no_grad():
            for Xb, yw, yp in loader:
                Xb, yw, yp = Xb.to(device), yw.to(device), yp.to(device)
                logit_w, logit_p = model(Xb)
                loss_w = bce(logit_w, yw)
                loss_p = bce(logit_p, yp)
                pw = torch.sigmoid(logit_w)
                pp = torch.sigmoid(logit_p)
                win_correct += ((pw > 0.5) == (yw > 0.5)).sum().item()
                pod_correct += ((pp > 0.5) == (yp > 0.5)).sum().item()
                total += Xb.size(0)
                win_loss_cum += loss_w.item() * Xb.size(0)
                pod_loss_cum += loss_p.item() * Xb.size(0)
        return (win_loss_cum / total, pod_loss_cum / total,
                win_correct / total, pod_correct / total)

    epochs = 80
    best_val = float('inf')
    patience = 10
    no_improve = 0
    for epoch in range(1, epochs + 1):
        model_torch.train()
        for Xb, yw, yp in train_loader:
            Xb, yw, yp = Xb.to(device), yw.to(device), yp.to(device)
            optimizer.zero_grad()
            logit_w, logit_p = model_torch(Xb)
            loss_w = bce(logit_w, yw)
            loss_p = bce(logit_p, yp)
            loss = loss_w + loss_p
            loss.backward()
            optimizer.step()
        win_l_val, pod_l_val, win_acc_val, pod_acc_val = evaluate(model_torch, val_loader)
        val_score = win_l_val + pod_l_val
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Val Loss W/P: {win_l_val:.4f}/{pod_l_val:.4f} | Acc W/P: {win_acc_val:.3f}/{pod_acc_val:.3f}")
        if val_score < best_val - 1e-4:
            best_val = val_score
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model_torch.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping.")
                break
    # Load best
    if 'best_state' in locals():
        model_torch.load_state_dict(best_state)

    # Store scaler params for inference
    torch_scaler = {"mean": feat_mean, "std": feat_std}

    def predict_future_race_torch(df, race_name, race_date, model, scaler, grid_dict=None, grid_weight: float = 0.3):
        """
        Torch inference mirroring sklearn flow with grid_weight boosting pole and decaying down the grid.
        """
        race_date = pd.to_datetime(race_date)
        past_df = df[df["race_date"] < race_date]
        drivers = past_df["driver_name"].dropna().unique()
        rows = []
        for driver in drivers:
            hist = past_df[past_df["driver_name"] == driver].sort_values("race_date").tail(5)
            if hist.empty:
                continue
            driver_form = hist["final_position"].astype(float).mean()
            driver_dnf_rate = (hist["status"].str.contains("DNF", case=False, na=False)).mean()
            grid_pos = grid_dict.get(driver) if grid_dict and driver in grid_dict else hist["grid_position"].dropna().mean()
            rows.append({
                "driver_name": driver,
                "grid_position": grid_pos,
                "driver_form": driver_form,
                "driver_dnf_rate": driver_dnf_rate,
            })
        if not rows:
            print("No drivers for prediction.")
            return pd.DataFrame()
        pred_df = pd.DataFrame(rows)
        # Impute
        pred_df['grid_position'] = pred_df['grid_position'].fillna(pred_df['grid_position'].median())
        pred_df['driver_form'] = pred_df['driver_form'].fillna(pred_df['driver_form'].median())
        pred_df['driver_dnf_rate'] = pred_df['driver_dnf_rate'].fillna(0.0)
        Xn = pred_df[["grid_position", "driver_form", "driver_dnf_rate"]]
        # Scale
        Xn_std = (Xn - scaler['mean']) / scaler['std'].replace(0, 1.0)
        Xt = torch.tensor(Xn_std.values, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logit_w, logit_p = model(Xt)
            # Use raw logits for grid adjustment
            logit_w_np = logit_w.cpu().numpy().ravel()
            podium_prob = torch.sigmoid(logit_p).cpu().numpy().ravel()
        # Grid-adjusted logits and softmax normalization
        grid_bonus = -grid_weight * (pred_df['grid_position'].values - 1.0)
        adj_logits = logit_w_np + grid_bonus
        exps = np.exp(adj_logits - adj_logits.max())
        win_prob = exps / exps.sum()
        pred_df['win_prob'] = win_prob
        pred_df['podium_prob'] = podium_prob
        print(f"Sum of win_prob (torch, grid-adjusted) after normalization: {pred_df['win_prob'].sum():.6f}")
        return pred_df.sort_values('win_prob', ascending=False)

    # Example inference (Italian GP date same as earlier example)
    preds_torch = predict_future_race_torch(df, "Italian GP", "2025-10-06", model_torch, torch_scaler, grid_weight=0.5)
    print("Top (PyTorch) Italian GP Predictions:")
    print(preds_torch.head())


Epoch 001 | Val Loss W/P: 0.6331/0.7260 | Acc W/P: 0.915/0.170
Epoch 005 | Val Loss W/P: 0.3034/0.4513 | Acc W/P: 0.915/0.830
Epoch 010 | Val Loss W/P: 0.1898/0.3727 | Acc W/P: 0.915/0.894
Epoch 015 | Val Loss W/P: 0.1518/0.4309 | Acc W/P: 0.915/0.915
Epoch 020 | Val Loss W/P: 0.1592/0.5034 | Acc W/P: 0.915/0.894
Early stopping.
Sum of win_prob (torch, grid-adjusted) after normalization: 1.000000
Top (PyTorch) Italian GP Predictions:
       driver_name  grid_position  driver_form  driver_dnf_rate  win_prob  \
0     Lando Norris            3.0          6.0              0.0  0.306921   
1   Max Verstappen            3.4          3.4              0.0  0.295786   
8    Oscar Piastri            3.4          5.4              0.0  0.260835   
2   George Russell            5.0          3.8              0.0  0.091706   
7  Charles Leclerc            4.8          7.8              0.0  0.043806   

   podium_prob  
0     0.501586  
1     0.526067  
8     0.506749  
2     0.484259  
7     0.348627